In [8]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import gc

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_NAME = "meta-llama/llama-3.2-3B" 
LAYER_ID = 16 
SEED = 42
N_SAMPLES = 1000 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

torch.manual_seed(SEED)
np.random.seed(SEED)

class CausalInterventionLab:
    def __init__(self, model_name, device):
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # CRITICAL: For batched inference/intervention, use Left Padding.
        # This ensures the 'last' token is the actual prompt end, not a pad token.
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=DTYPE,
            device_map="auto",
            attn_implementation="eager" 
        )
        self.device = device
        self.layer_id = LAYER_ID
        
        # Mechanisms
        self.steering_vector = None
        self.bypass_probe = None
        self.factual_pca = None
        self.factual_mean = None
        self.factual_components = None
        self.hooks = []

    def clear_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def get_layer_module(self, layer_idx):
        return self.model.model.layers[layer_idx]

    def _get_hidden(self, output):
        """Safely extract hidden states whether output is tuple or tensor."""
        if isinstance(output, tuple):
            return output[0]
        return output

    # ==========================================
    # DATA PREPARATION
    # ==========================================
    def format_prompt(self, text):
        return f"Q: {text}\nA:"

    def prepare_dataset(self):
        print("Preparing Datasets...")
        half_n = N_SAMPLES // 2
        
        ds_truth = load_dataset("truthful_qa", "generation", split="validation")
        truthful_raw = ds_truth.select(range(half_n))['question']
        truthful_prompts = [self.format_prompt(q) for q in truthful_raw]
        
        ds_pop = load_dataset("akariasai/PopQA", split="test")
        pop_sorted = ds_pop.sort("s_pop") 
        pop_raw = pop_sorted.select(range(half_n))['question']
        pop_prompts = [self.format_prompt(q) for q in pop_raw]
        
        print(f"Loaded {len(truthful_prompts)} Factual and {len(pop_prompts)} Uncertain.")
        return truthful_prompts, pop_prompts

    # ==========================================
    # STEP 1: ANALYSIS & MECHANISM EXTRACTION
    # ==========================================
    def collect_activations(self, prompts, batch_size=4):
        activations = []
        
        def cache_hook(module, input, output):
            # Safe extraction
            hidden = self._get_hidden(output) # (Batch, Seq, Dim)
            # Take last token
            activations.append(hidden[:, -1, :].detach().float().cpu())
            
        handle = self.get_layer_module(self.layer_id).register_forward_hook(cache_hook)
        
        print("Collecting activations...")
        for i in tqdm(range(0, len(prompts), batch_size)):
            batch = prompts[i:i+batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(self.device)
            with torch.no_grad():
                self.model(**inputs)
        
        handle.remove()
        return torch.cat(activations, dim=0).numpy()

    def train_mechanisms(self, factual_acts, uncertain_acts):
        print("\n--- Training Intervention Mechanisms ---")
        
        # 1. Steering Vector
        self.steering_vector = np.mean(uncertain_acts, axis=0) - np.mean(factual_acts, axis=0)
        self.steering_vector = torch.tensor(self.steering_vector, device=self.device, dtype=DTYPE)
        print(f"[Steering] Vector norm: {torch.norm(self.steering_vector):.2f}")

        # 2. Probe
        X = np.concatenate([factual_acts, uncertain_acts])
        y = np.concatenate([np.zeros(len(factual_acts)), np.ones(len(uncertain_acts))])
        
        self.bypass_probe = LogisticRegression(max_iter=1000).fit(X, y)
        acc = self.bypass_probe.score(X, y)
        print(f"[Bypass] Probe accuracy: {acc:.2%}")

        # 3. Manifold (PCA)
        self.factual_pca = PCA(n_components=0.95)
        self.factual_pca.fit(factual_acts)
        
        self.factual_mean = torch.tensor(self.factual_pca.mean_, device=self.device, dtype=DTYPE)
        self.factual_components = torch.tensor(self.factual_pca.components_, device=self.device, dtype=DTYPE)
        print(f"[Manifold] Factual subspace dimensions: {self.factual_pca.n_components_}")

    # ==========================================
    # INTERVENTION 1: RESERVOIR STEERING
    # ==========================================
    def run_steering_intervention(self, prompt, alpha=2.0):
        self.clear_hooks()
        
        def steering_hook(module, input, output):
            # Depending on return type, we might need to reconstruct the tuple
            is_tuple = isinstance(output, tuple)
            hidden = output[0] if is_tuple else output
            
            # Injection
            hidden[:, -1, :] += alpha * self.steering_vector
            
            return (hidden,) + output[1:] if is_tuple else hidden
            
        handle = self.get_layer_module(self.layer_id).register_forward_hook(steering_hook)
        self.hooks.append(handle)
        
        return self._generate(prompt)

    # ==========================================
    # INTERVENTION 2: NULL SPACE BYPASS
    # ==========================================
    def run_bypass_intervention(self, prompt, refusal_gain=5.0):
        self.clear_hooks()
        
        # Token for " Unsure" (Note the space for Llama tokenizer)
        # You might also try " I" (id: 40) or " not" (id: 459) depending on desired output
        target_token_id = self.tokenizer.encode(" Unsure")[1] # [1] usually gets the word, [0] is BOS
        
        captured = {}

        def capture_hook(module, input, output):
            hidden = self._get_hidden(output)
            captured['last_hidden'] = hidden[:, -1, :].detach().float().cpu().numpy()
            return output

        handle = self.get_layer_module(self.layer_id).register_forward_hook(capture_hook)
        self.hooks.append(handle)

        class BypassLogitsProcessor(LogitsProcessor):
            def __init__(self, probe, gain, target_id):
                self.probe = probe
                self.gain = gain
                self.target_id = target_id
                self.checked = False

            def __call__(self, input_ids, scores):
                if not self.checked and 'last_hidden' in captured:
                    prob_halluc = self.probe.predict_proba(captured['last_hidden'])[:, 1]
                    intervention_tensor = torch.tensor(prob_halluc, device=scores.device)
                    scores[:, self.target_id] += intervention_tensor * self.gain
                    self.checked = True
                return scores

        lp = [BypassLogitsProcessor(self.bypass_probe, refusal_gain, target_token_id)]
        
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=30, 
                do_sample=True, 
                temperature=0.7,
                logits_processor=lp,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        self.clear_hooks()
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ==========================================
    # INTERVENTION 3: MANIFOLD COMPRESSION
    # ==========================================
    def run_manifold_intervention(self, prompt):
        self.clear_hooks()
        
        def compression_hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden = output[0] if is_tuple else output
            
            # Soft Clamp: Mix the original hidden state with the projected one
            # 80% Original, 20% Projected. Preserves syntax, reduces "Sprawl".
            centered = hidden - self.factual_mean
            proj = torch.matmul(centered, self.factual_components.T)
            recon = torch.matmul(proj, self.factual_components) + self.factual_mean
            
            final = 0.8 * hidden + 0.2 * recon 
            
            return (final.to(hidden.dtype),) + output[1:] if is_tuple else final

        handle = self.get_layer_module(self.layer_id).register_forward_hook(compression_hook)
        self.hooks.append(handle)
        
        return self._generate(prompt)

    def _generate(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=40, 
                do_sample=True, 
                temperature=0.1, 
                pad_token_id=self.tokenizer.eos_token_id
            )
        output_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return output_text.replace(prompt, "").strip()


# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    lab = CausalInterventionLab(MODEL_NAME, DEVICE)
    
    fact_prompts, uncertain_prompts = lab.prepare_dataset()
    
    # Using 200 for training mechanisms
    print("Collecting training data for mechanisms...")
    f_acts = lab.collect_activations(fact_prompts[:200])
    u_acts = lab.collect_activations(uncertain_prompts[:200])
    
    lab.train_mechanisms(f_acts, u_acts)
    
    print("\n\n=== RUNNING CAUSAL INTERVENTIONS ON BASE MODEL ===")
    
    # 1. STEERING
    fact_q = lab.format_prompt("The capital of France is")
    print(f"\n[Test 1] Reservoir Steering (Add Uncertainty)")
    print(f"Prompt: {fact_q.strip()}")
    print(f"Baseline:     {lab._generate(fact_q)}")
    print(f"Intervention: {lab.run_steering_intervention(fact_q, alpha=2.0)}")
    
    # 2. BYPASS
    uncert_q = uncertain_prompts[50] 
    print(f"\n[Test 2] Null Space Bypass (Force 'Unsure')")
    print(f"Prompt: {uncert_q.strip()}")
    print(f"Baseline:     {lab._generate(uncert_q)}")
    print(f"Intervention: {lab.run_bypass_intervention(uncert_q, refusal_gain=25.0)}")
    
    # 3. MANIFOLD
    print(f"\n[Test 3] Manifold Compression (Remove high-LID)")
    print(f"Prompt: {uncert_q.strip()}")
    print(f"Intervention: {lab.run_manifold_intervention(uncert_q)}")
    
    print("\nDone.")

Loading Qwen/Qwen2.5-3B...


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.76it/s]


Preparing Datasets...


Repo card metadata block was not found. Setting CardData to empty.


Loaded 500 Factual and 500 Uncertain.


100%|██████████| 50/50 [00:02<00:00, 21.23it/s]


100%|██████████| 50/50 [00:02<00:00, 21.96it/s]



--- Training Intervention Mechanisms ---
[Steering] Vector norm: 12.81
[Bypass] Probe accuracy: 100.00%
[Manifold] Factual subspace dimensions: 78


=== RUNNING CAUSAL INTERVENTIONS ON BASE MODEL ===

[Test 1] Reservoir Steering (Add Uncertainty)
Prompt: Q: The capital of France is
A:
Baseline:     Paris
Intervention: Paris

[Test 2] Null Space Bypass (Force 'Unsure')
Prompt: Q: In what country is Goreme?
A:
Baseline:     Turkey


IndexError: list index out of range

In [9]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import gc
import os

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_NAME = "meta-llama/llama-3.2-3B" 
LAYER_ID = 16 
SEED = 42
TOTAL_SAMPLES = 1000  # 500 Factual, 500 Uncertain
TRAIN_SIZE = 200      # Use 100 of each for training mechanisms
TEST_SIZE = TOTAL_SAMPLES - TRAIN_SIZE
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Parameters tuned based on your previous runs
STEERING_ALPHA = 2.0      # Lowered slightly to prevent total collapse
BYPASS_GAIN = 20.0        # Kept high to force "Unsure"
MANIFOLD_MIX = 0.2        # 20% Projection, 80% Original (Soft Clamp)

torch.manual_seed(SEED)
np.random.seed(SEED)

class CausalExperiment:
    def __init__(self, model_name, device):
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=DTYPE,
            device_map="auto",
            attn_implementation="eager" 
        )
        self.device = device
        self.layer_id = LAYER_ID
        self.hooks = []
        
        # Mechanisms
        self.steering_vector = None
        self.bypass_probe = None
        self.factual_pca = None
        self.factual_mean = None
        self.factual_components = None

    def clear_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def _get_hidden(self, output):
        return output[0] if isinstance(output, tuple) else output

    def format_prompt(self, text):
        return f"Q: {text}\nA:"

    # ==========================================
    # DATASET PREP
    # ==========================================
    def prepare_dataset(self):
        print("Preparing Datasets...")
        half_n = TOTAL_SAMPLES // 2
        
        # Factual: TruthfulQA
        ds_truth = load_dataset("truthful_qa", "generation", split="validation")
        truthful_raw = ds_truth.select(range(half_n))['question']
        
        # Uncertain: PopQA (Low Popularity)
        ds_pop = load_dataset("akariasai/PopQA", split="test")
        pop_sorted = ds_pop.sort("s_pop") 
        pop_raw = pop_sorted.select(range(half_n))['question']
        
        dataset = []
        
        # Add Factual
        for q in truthful_raw:
            dataset.append({
                "type": "Factual",
                "prompt_text": q,
                "formatted_prompt": self.format_prompt(q)
            })
            
        # Add Uncertain
        for q in pop_raw:
            dataset.append({
                "type": "Uncertain",
                "prompt_text": q,
                "formatted_prompt": self.format_prompt(q)
            })
            
        df = pd.DataFrame(dataset)
        # Shuffle
        df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        return df

    # ==========================================
    # MECHANISM TRAINING
    # ==========================================
    def collect_activations(self, prompts, batch_size=4):
        activations = []
        def cache_hook(module, input, output):
            hidden = self._get_hidden(output)
            activations.append(hidden[:, -1, :].detach().float().cpu())
            
        handle = self.model.model.layers[self.layer_id].register_forward_hook(cache_hook)
        
        for i in tqdm(range(0, len(prompts), batch_size), desc="Collecting Acts"):
            batch = prompts[i:i+batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(self.device)
            with torch.no_grad():
                self.model(**inputs)
        
        handle.remove()
        return torch.cat(activations, dim=0).numpy()

    def train_mechanisms(self, train_df):
        print("\n--- Training Mechanisms ---")
        f_df = train_df[train_df['type'] == 'Factual']
        u_df = train_df[train_df['type'] == 'Uncertain']
        
        f_acts = self.collect_activations(f_df['formatted_prompt'].tolist())
        u_acts = self.collect_activations(u_df['formatted_prompt'].tolist())
        
        # 1. Steering
        self.steering_vector = np.mean(u_acts, axis=0) - np.mean(f_acts, axis=0)
        self.steering_vector = torch.tensor(self.steering_vector, device=self.device, dtype=DTYPE)
        
        # 2. Probe
        X = np.concatenate([f_acts, u_acts])
        y = np.concatenate([np.zeros(len(f_acts)), np.ones(len(u_acts))])
        self.bypass_probe = LogisticRegression(max_iter=1000).fit(X, y)
        print(f"Probe Acc: {self.bypass_probe.score(X, y):.2%}")
        
        # 3. Manifold
        self.factual_pca = PCA(n_components=0.95)
        self.factual_pca.fit(f_acts)
        self.factual_mean = torch.tensor(self.factual_pca.mean_, device=self.device, dtype=DTYPE)
        self.factual_components = torch.tensor(self.factual_pca.components_, device=self.device, dtype=DTYPE)

    # ==========================================
    # INTERVENTIONS
    # ==========================================
    def _generate(self, prompt, logits_processor=None):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=40, 
                do_sample=True, 
                temperature=0.1, 
                logits_processor=logits_processor,
                pad_token_id=self.tokenizer.eos_token_id
            )
        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return full_text.replace(prompt, "").strip()

    def run_baseline(self, prompt):
        return self._generate(prompt)

    def run_steering(self, prompt):
        self.clear_hooks()
        def hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden = output[0] if is_tuple else output
            hidden[:, -1, :] += STEERING_ALPHA * self.steering_vector
            return (hidden,) + output[1:] if is_tuple else hidden
        
        h = self.model.model.layers[self.layer_id].register_forward_hook(hook)
        self.hooks.append(h)
        res = self._generate(prompt)
        self.clear_hooks()
        return res

    def run_bypass(self, prompt):
        self.clear_hooks()
        # " Unsure" token
        target_token_id = self.tokenizer.encode(" Unsure")[1]
        
        captured = {}
        def capture_hook(module, input, output):
            hidden = self._get_hidden(output)
            captured['last_hidden'] = hidden[:, -1, :].detach().float().cpu().numpy()
            return output
        
        self.hooks.append(self.model.model.layers[self.layer_id].register_forward_hook(capture_hook))

        class BypassLP(LogitsProcessor):
            def __init__(self, probe, gain, target_id):
                self.probe = probe
                self.gain = gain
                self.target_id = target_id
                self.checked = False
            def __call__(self, input_ids, scores):
                if not self.checked and 'last_hidden' in captured:
                    prob = self.probe.predict_proba(captured['last_hidden'])[:, 1]
                    scores[:, self.target_id] += torch.tensor(prob, device=scores.device) * self.gain
                    self.checked = True
                return scores

        lp = [BypassLP(self.bypass_probe, BYPASS_GAIN, target_token_id)]
        res = self._generate(prompt, logits_processor=lp)
        self.clear_hooks()
        return res

    def run_manifold(self, prompt):
        self.clear_hooks()
        def hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden = output[0] if is_tuple else output
            
            # Soft Clamp (Mixing)
            centered = hidden - self.factual_mean
            proj = torch.matmul(centered, self.factual_components.T)
            recon = torch.matmul(proj, self.factual_components) + self.factual_mean
            
            # Mix: (1 - MANIFOLD_MIX)*Original + MANIFOLD_MIX*Reconstructed
            # Wait, typically we want to REPLACE the noisy part with structure.
            # Let's project heavily: 20% Original (Noise), 80% Structure
            # Note: In your "The The The" error, you did 100% structure. 
            # Let's try preserving 80% original to keep syntax, but cleaning 20%.
            final = (1 - MANIFOLD_MIX) * hidden + MANIFOLD_MIX * recon
            
            return (final.to(hidden.dtype),) + output[1:] if is_tuple else final

        h = self.model.model.layers[self.layer_id].register_forward_hook(hook)
        self.hooks.append(h)
        res = self._generate(prompt)
        self.clear_hooks()
        return res

# ==========================================
# EVALUATION & METRICS
# ==========================================
def analyze_output(text):
    text_lower = text.lower()
    
    # 1. Detect Refusal
    refusal_keywords = ["unsure", "i don't know", "i do not know", "not sure", "sorry", "uncertain"]
    is_refusal = any(k in text_lower for k in refusal_keywords)
    
    # 2. Detect Repetition Loop (naive check for repeating 4-grams)
    words = text_lower.split()
    is_loop = False
    if len(words) > 10:
        # Check if the last 5 words appeared identically just before
        tail = words[-5:]
        body = words[:-5]
        # Quick heuristic
        for i in range(len(body) - 5):
            if body[i:i+5] == tail:
                is_loop = True
                break
                
    return is_refusal, is_loop

# ==========================================
# MAIN LOOP
# ==========================================
if __name__ == "__main__":
    exp = CausalExperiment(MODEL_NAME, DEVICE)
    
    # 1. Data
    full_df = exp.prepare_dataset()
    train_df = full_df.iloc[:TRAIN_SIZE]
    test_df = full_df.iloc[TRAIN_SIZE:]
    
    # 2. Train
    exp.train_mechanisms(train_df)
    
    # 3. Test Loop
    results = []
    
    print(f"\nRunning tests on {len(test_df)} samples...")
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        prompt = row['formatted_prompt']
        p_type = row['type']
        
        # A. Baseline
        base_text = exp.run_baseline(prompt)
        base_refusal, base_loop = analyze_output(base_text)
        
        res_entry = {
            "id": idx,
            "type": p_type,
            "prompt": row['prompt_text'],
            "baseline": base_text,
            "base_refusal": base_refusal,
            "base_loop": base_loop,
            "intervention_name": "None",
            "intervention_text": None,
            "int_refusal": None,
            "int_loop": None
        }
        
        # B. Apply Specific Interventions based on Type
        
        if p_type == "Factual":
            # Test 1: Steering (Try to break it)
            int_text = exp.run_steering(prompt)
            int_refusal, int_loop = analyze_output(int_text)
            
            # Save Steering Entry
            s_entry = res_entry.copy()
            s_entry["intervention_name"] = "Steering"
            s_entry["intervention_text"] = int_text
            s_entry["int_refusal"] = int_refusal
            s_entry["int_loop"] = int_loop
            results.append(s_entry)
            
        elif p_type == "Uncertain":
            # Test 2: Bypass (Try to fix it with Refusal)
            bypass_text = exp.run_bypass(prompt)
            b_refusal, b_loop = analyze_output(bypass_text)
            
            b_entry = res_entry.copy()
            b_entry["intervention_name"] = "Bypass"
            b_entry["intervention_text"] = bypass_text
            b_entry["int_refusal"] = b_refusal
            b_entry["int_loop"] = b_loop
            results.append(b_entry)
            
            # Test 3: Manifold (Try to fix structure)
            man_text = exp.run_manifold(prompt)
            m_refusal, m_loop = analyze_output(man_text)
            
            m_entry = res_entry.copy()
            m_entry["intervention_name"] = "Manifold"
            m_entry["intervention_text"] = man_text
            m_entry["int_refusal"] = m_refusal
            m_entry["int_loop"] = m_loop
            results.append(m_entry)

    # 4. Save
    res_df = pd.DataFrame(results)
    res_df.to_csv("intervention_results.csv", index=False)
    
    print("\n=== SUMMARY STATISTICS ===")
    
    # Analyze Steering Effectiveness (Factual)
    steer_df = res_df[res_df["intervention_name"] == "Steering"]
    changed = (steer_df["baseline"] != steer_df["intervention_text"]).mean()
    loops = steer_df["int_loop"].mean()
    print(f"[Steering] Changed Output: {changed:.2%}, Loops Induced: {loops:.2%}")
    
    # Analyze Bypass Effectiveness (Uncertain)
    bypass_df = res_df[res_df["intervention_name"] == "Bypass"]
    base_ref_rate = bypass_df["base_refusal"].mean()
    int_ref_rate = bypass_df["int_refusal"].mean()
    print(f"[Bypass] Refusal Rate: {base_ref_rate:.2%} -> {int_ref_rate:.2%}")
    
    # Analyze Manifold Effectiveness (Uncertain)
    man_df = res_df[res_df["intervention_name"] == "Manifold"]
    base_loop_rate = man_df["base_loop"].mean()
    int_loop_rate = man_df["int_loop"].mean()
    print(f"[Manifold] Loop Rate: {base_loop_rate:.2%} -> {int_loop_rate:.2%}")
    
    print("\nResults saved to 'intervention_results.csv'")

Loading Qwen/Qwen2.5-3B...


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.67it/s]


Preparing Datasets...


Repo card metadata block was not found. Setting CardData to empty.



--- Training Mechanisms ---


Probe Acc: 100.00%

Running tests on 800 samples...


  0%|          | 1/800 [00:00<05:10,  2.58it/s]


IndexError: list index out of range

In [10]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import gc
import os

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_NAME = "Qwen/Qwen2.5-3B" 
LAYER_ID = 16 
SEED = 42
TOTAL_SAMPLES = 1000  # 500 Factual, 500 Uncertain
TRAIN_SIZE = 200      # Use 100 of each for training mechanisms
TEST_SIZE = TOTAL_SAMPLES - TRAIN_SIZE
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Parameters tuned based on your previous runs
STEERING_ALPHA = 2.0      # Lowered slightly to prevent total collapse
BYPASS_GAIN = 20.0        # Kept high to force "Unsure"
MANIFOLD_MIX = 0.2        # 20% Projection, 80% Original (Soft Clamp)

torch.manual_seed(SEED)
np.random.seed(SEED)

class CausalExperiment:
    def __init__(self, model_name, device):
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=DTYPE,
            device_map="auto",
            attn_implementation="eager" 
        )
        self.device = device
        self.layer_id = LAYER_ID
        self.hooks = []
        
        # Mechanisms
        self.steering_vector = None
        self.bypass_probe = None
        self.factual_pca = None
        self.factual_mean = None
        self.factual_components = None

    def clear_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def _get_hidden(self, output):
        return output[0] if isinstance(output, tuple) else output

    def format_prompt(self, text):
        return f"Q: {text}\nA:"

    # ==========================================
    # DATASET PREP
    # ==========================================
    def prepare_dataset(self):
        print("Preparing Datasets...")
        half_n = TOTAL_SAMPLES // 2
        
        # Factual: TruthfulQA
        ds_truth = load_dataset("truthful_qa", "generation", split="validation")
        truthful_raw = ds_truth.select(range(half_n))['question']
        
        # Uncertain: PopQA (Low Popularity)
        ds_pop = load_dataset("akariasai/PopQA", split="test")
        pop_sorted = ds_pop.sort("s_pop") 
        pop_raw = pop_sorted.select(range(half_n))['question']
        
        dataset = []
        
        # Add Factual
        for q in truthful_raw:
            dataset.append({
                "type": "Factual",
                "prompt_text": q,
                "formatted_prompt": self.format_prompt(q)
            })
            
        # Add Uncertain
        for q in pop_raw:
            dataset.append({
                "type": "Uncertain",
                "prompt_text": q,
                "formatted_prompt": self.format_prompt(q)
            })
            
        df = pd.DataFrame(dataset)
        # Shuffle
        df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        return df

    # ==========================================
    # MECHANISM TRAINING
    # ==========================================
    def collect_activations(self, prompts, batch_size=4):
        activations = []
        def cache_hook(module, input, output):
            hidden = self._get_hidden(output)
            activations.append(hidden[:, -1, :].detach().float().cpu())
            
        handle = self.model.model.layers[self.layer_id].register_forward_hook(cache_hook)
        
        for i in tqdm(range(0, len(prompts), batch_size), desc="Collecting Acts"):
            batch = prompts[i:i+batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(self.device)
            with torch.no_grad():
                self.model(**inputs)
        
        handle.remove()
        return torch.cat(activations, dim=0).numpy()

    def train_mechanisms(self, train_df):
        print("\n--- Training Mechanisms ---")
        f_df = train_df[train_df['type'] == 'Factual']
        u_df = train_df[train_df['type'] == 'Uncertain']
        
        f_acts = self.collect_activations(f_df['formatted_prompt'].tolist())
        u_acts = self.collect_activations(u_df['formatted_prompt'].tolist())
        
        # 1. Steering
        self.steering_vector = np.mean(u_acts, axis=0) - np.mean(f_acts, axis=0)
        self.steering_vector = torch.tensor(self.steering_vector, device=self.device, dtype=DTYPE)
        
        # 2. Probe
        X = np.concatenate([f_acts, u_acts])
        y = np.concatenate([np.zeros(len(f_acts)), np.ones(len(u_acts))])
        self.bypass_probe = LogisticRegression(max_iter=1000).fit(X, y)
        print(f"Probe Acc: {self.bypass_probe.score(X, y):.2%}")
        
        # 3. Manifold
        self.factual_pca = PCA(n_components=0.95)
        self.factual_pca.fit(f_acts)
        self.factual_mean = torch.tensor(self.factual_pca.mean_, device=self.device, dtype=DTYPE)
        self.factual_components = torch.tensor(self.factual_pca.components_, device=self.device, dtype=DTYPE)

    # ==========================================
    # INTERVENTIONS
    # ==========================================
    def _generate(self, prompt, logits_processor=None):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=40, 
                do_sample=True, 
                temperature=0.1, 
                logits_processor=logits_processor,
                pad_token_id=self.tokenizer.eos_token_id
            )
        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return full_text.replace(prompt, "").strip()

    def run_baseline(self, prompt):
        return self._generate(prompt)

    def run_steering(self, prompt):
        self.clear_hooks()
        def hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden = output[0] if is_tuple else output
            hidden[:, -1, :] += STEERING_ALPHA * self.steering_vector
            return (hidden,) + output[1:] if is_tuple else hidden
        
        h = self.model.model.layers[self.layer_id].register_forward_hook(hook)
        self.hooks.append(h)
        res = self._generate(prompt)
        self.clear_hooks()
        return res

    def run_bypass(self, prompt):
            self.clear_hooks()
            
            # FIX: Use [-1] to get the last token. 
            # This works whether a BOS token exists (Llama) or not (Qwen).
            # You might also want to try " I" or " Sorry" for Qwen.
            target_token_id = self.tokenizer.encode(" Unsure")[-1]
            
            captured = {}
            def capture_hook(module, input, output):
                hidden = self._get_hidden(output)
                captured['last_hidden'] = hidden[:, -1, :].detach().float().cpu().numpy()
                return output
            
            self.hooks.append(self.model.model.layers[self.layer_id].register_forward_hook(capture_hook))
    
            class BypassLP(LogitsProcessor):
                def __init__(self, probe, gain, target_id):
                    self.probe = probe
                    self.gain = gain
                    self.target_id = target_id
                    self.checked = False
                def __call__(self, input_ids, scores):
                    if not self.checked and 'last_hidden' in captured:
                        prob = self.probe.predict_proba(captured['last_hidden'])[:, 1]
                        # Convert prob to tensor to match scores device
                        intervention = torch.tensor(prob, device=scores.device, dtype=scores.dtype)
                        scores[:, self.target_id] += intervention * self.gain
                        self.checked = True
                    return scores
    
            lp = [BypassLP(self.bypass_probe, BYPASS_GAIN, target_token_id)]
            res = self._generate(prompt, logits_processor=lp)
            self.clear_hooks()
            return res

    def run_manifold(self, prompt):
        self.clear_hooks()
        def hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden = output[0] if is_tuple else output
            
            # Soft Clamp (Mixing)
            centered = hidden - self.factual_mean
            proj = torch.matmul(centered, self.factual_components.T)
            recon = torch.matmul(proj, self.factual_components) + self.factual_mean
            
            # Mix: (1 - MANIFOLD_MIX)*Original + MANIFOLD_MIX*Reconstructed
            # Wait, typically we want to REPLACE the noisy part with structure.
            # Let's project heavily: 20% Original (Noise), 80% Structure
            # Note: In your "The The The" error, you did 100% structure. 
            # Let's try preserving 80% original to keep syntax, but cleaning 20%.
            final = (1 - MANIFOLD_MIX) * hidden + MANIFOLD_MIX * recon
            
            return (final.to(hidden.dtype),) + output[1:] if is_tuple else final

        h = self.model.model.layers[self.layer_id].register_forward_hook(hook)
        self.hooks.append(h)
        res = self._generate(prompt)
        self.clear_hooks()
        return res

# ==========================================
# EVALUATION & METRICS
# ==========================================
def analyze_output(text):
    text_lower = text.lower()
    
    # 1. Detect Refusal
    refusal_keywords = ["unsure", "i don't know", "i do not know", "not sure", "sorry", "uncertain"]
    is_refusal = any(k in text_lower for k in refusal_keywords)
    
    # 2. Detect Repetition Loop (naive check for repeating 4-grams)
    words = text_lower.split()
    is_loop = False
    if len(words) > 10:
        # Check if the last 5 words appeared identically just before
        tail = words[-5:]
        body = words[:-5]
        # Quick heuristic
        for i in range(len(body) - 5):
            if body[i:i+5] == tail:
                is_loop = True
                break
                
    return is_refusal, is_loop

# ==========================================
# MAIN LOOP
# ==========================================
if __name__ == "__main__":
    exp = CausalExperiment(MODEL_NAME, DEVICE)
    
    # 1. Data
    full_df = exp.prepare_dataset()
    train_df = full_df.iloc[:TRAIN_SIZE]
    test_df = full_df.iloc[TRAIN_SIZE:]
    
    # 2. Train
    exp.train_mechanisms(train_df)
    
    # 3. Test Loop
    results = []
    
    print(f"\nRunning tests on {len(test_df)} samples...")
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        prompt = row['formatted_prompt']
        p_type = row['type']
        
        # A. Baseline
        base_text = exp.run_baseline(prompt)
        base_refusal, base_loop = analyze_output(base_text)
        
        res_entry = {
            "id": idx,
            "type": p_type,
            "prompt": row['prompt_text'],
            "baseline": base_text,
            "base_refusal": base_refusal,
            "base_loop": base_loop,
            "intervention_name": "None",
            "intervention_text": None,
            "int_refusal": None,
            "int_loop": None
        }
        
        # B. Apply Specific Interventions based on Type
        
        if p_type == "Factual":
            # Test 1: Steering (Try to break it)
            int_text = exp.run_steering(prompt)
            int_refusal, int_loop = analyze_output(int_text)
            
            # Save Steering Entry
            s_entry = res_entry.copy()
            s_entry["intervention_name"] = "Steering"
            s_entry["intervention_text"] = int_text
            s_entry["int_refusal"] = int_refusal
            s_entry["int_loop"] = int_loop
            results.append(s_entry)
            
        elif p_type == "Uncertain":
            # Test 2: Bypass (Try to fix it with Refusal)
            bypass_text = exp.run_bypass(prompt)
            b_refusal, b_loop = analyze_output(bypass_text)
            
            b_entry = res_entry.copy()
            b_entry["intervention_name"] = "Bypass"
            b_entry["intervention_text"] = bypass_text
            b_entry["int_refusal"] = b_refusal
            b_entry["int_loop"] = b_loop
            results.append(b_entry)
            
            # Test 3: Manifold (Try to fix structure)
            man_text = exp.run_manifold(prompt)
            m_refusal, m_loop = analyze_output(man_text)
            
            m_entry = res_entry.copy()
            m_entry["intervention_name"] = "Manifold"
            m_entry["intervention_text"] = man_text
            m_entry["int_refusal"] = m_refusal
            m_entry["int_loop"] = m_loop
            results.append(m_entry)

    # 4. Save
    res_df = pd.DataFrame(results)
    res_df.to_csv("intervention_results.csv", index=False)
    
    print("\n=== SUMMARY STATISTICS ===")
    
    # Analyze Steering Effectiveness (Factual)
    steer_df = res_df[res_df["intervention_name"] == "Steering"]
    changed = (steer_df["baseline"] != steer_df["intervention_text"]).mean()
    loops = steer_df["int_loop"].mean()
    print(f"[Steering] Changed Output: {changed:.2%}, Loops Induced: {loops:.2%}")
    
    # Analyze Bypass Effectiveness (Uncertain)
    bypass_df = res_df[res_df["intervention_name"] == "Bypass"]
    base_ref_rate = bypass_df["base_refusal"].mean()
    int_ref_rate = bypass_df["int_refusal"].mean()
    print(f"[Bypass] Refusal Rate: {base_ref_rate:.2%} -> {int_ref_rate:.2%}")
    
    # Analyze Manifold Effectiveness (Uncertain)
    man_df = res_df[res_df["intervention_name"] == "Manifold"]
    base_loop_rate = man_df["base_loop"].mean()
    int_loop_rate = man_df["int_loop"].mean()
    print(f"[Manifold] Loop Rate: {base_loop_rate:.2%} -> {int_loop_rate:.2%}")
    
    print("\nResults saved to 'intervention_results.csv'")

Loading Qwen/Qwen2.5-3B...


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.75it/s]


Preparing Datasets...


Repo card metadata block was not found. Setting CardData to empty.



--- Training Mechanisms ---


Probe Acc: 100.00%

Running tests on 800 samples...


100%|██████████| 800/800 [34:33<00:00,  2.59s/it]


=== SUMMARY STATISTICS ===
[Steering] Changed Output: 89.36%, Loops Induced: 16.09%
[Bypass] Refusal Rate: 0.00% -> 99.75%
[Manifold] Loop Rate: 1.77% -> 1.77%

Results saved to 'intervention_results.csv'
